# python203 — Tokenized Reward System for Financial Backtesting

> Master 203 — Alternative Finance · Paris Dauphine-PSL  
> Amandine Esteve · Edouard Lesbre · Benoit Faivre

This notebook demonstrates the full pipeline:

```
Trading strategy ──► Backtest ──► Score (0 – 10) ──► Token transfer (on-chain)
```

---

## 0 · Setup

In [ ]:
import os
import warnings
from datetime import datetime

import pandas as pd

warnings.filterwarnings("ignore")

from python203 import VolumeBacktest, BacktestRating, TokenTransfer

print("Package loaded successfully.")

---
## 1 · Run the backtest

`VolumeBacktest` fetches daily OHLCV data for a universe of 10 tech stocks and applies a **volume-momentum strategy**:

| Signal | Condition |
|--------|-----------|
| **Buy  (+1)** | Daily volume > 1.5 × 15-day average |
| **Sell (−1)** | Daily volume < 0.5 × 15-day average |
| **Hold ( 0)** | Otherwise |

We backtest over the full year **2023**, starting with **£1 000 000** of capital.

In [ ]:
bt = VolumeBacktest(
    initial_date=datetime(2023, 1, 1),
    final_date=datetime(2023, 12, 31),
    verbose=False,
)

print(f"Universe ({len(bt.universe)} tickers): {', '.join(bt.universe)}")
print(f"Period  : {bt.initial_date.date()} → {bt.final_date.date()}")
print(f"Capital : £{bt.initial_cash:,}")

In [ ]:
metrics = bt.run()

print("\n── Backtest results ──────────────────────────")
print(f"  Final portfolio value : £{metrics['Final Portfolio Value']:>15,.2f}")
print(f"  Sharpe ratio          : {metrics['Sharpe Ratio']:>15.4f}")
print(f"  Maximum drawdown      : {metrics['Maximum Drawdown']:>15.2%}")
print("──────────────────────────────────────────────")

---
## 2 · Score the strategy

`BacktestRating` maps the three metrics to a single score in **[0, 10]** using the following weights:

| Component | Weight | Formula |
|-----------|--------|---------|
| Portfolio growth | **50 %** | `min(5, (value / 1_000_000 − 1) × 5)` |
| Sharpe ratio | **25 %** | `min(2.5, sharpe / 1.0 × 2.5)` |
| Max drawdown | **25 %** | `min(2.5, (1 − drawdown / −0.4) × 2.5)` |

In [ ]:
rater = BacktestRating()
score = rater.rate_backtest(metrics)

# Breakdown per component
sharpe   = metrics["Sharpe Ratio"]
drawdown = metrics["Maximum Drawdown"]
value    = metrics["Final Portfolio Value"]

sharpe_score   = min(2.5, max(0.0, sharpe / rater.sharpe_threshold * 2.5))
dd_score       = min(2.5, max(0.0, (1 - drawdown / rater.max_drawdown_threshold) * 2.5)) \
                 if drawdown > rater.max_drawdown_threshold else 0.0
growth_score   = min(5.0, max(0.0, (value / 1_000_000 - 1) * 5))

print("── Score breakdown ───────────────────────────")
print(f"  Portfolio growth  (×50%) : {growth_score:.2f} / 5.00")
print(f"  Sharpe ratio      (×25%) : {sharpe_score:.2f} / 2.50")
print(f"  Max drawdown      (×25%) : {dd_score:.2f} / 2.50")
print("  ──────────────────────────────────────")
print(f"  TOTAL SCORE              : {score:.2f} / 10.00")
print("──────────────────────────────────────────────")

---
## 3 · Reward with ERC-20 tokens (on-chain)

The **Master203token (203T)** is an ERC-20 token deployed on the **Ethereum Sepolia testnet** with a fixed supply of **203 tokens**.

| Property | Value |
|----------|-------|
| Name | Master203token |
| Symbol | 203T |
| Total supply | 203 |
| Contract | `0xA0f0a2D53b3476c50F2Cf24307F8a1Cd3c758254` |
| Network | Ethereum Sepolia testnet |

[View on Sepolia Etherscan →](https://sepolia.etherscan.io/token/0xA0f0a2D53b3476c50F2Cf24307F8a1Cd3c758254)

The transfer sends exactly **`score` tokens** (0–10) to the recipient's wallet.

In [ ]:
# ── Token transfer ──────────────────────────────────────────────────────────
# Requires two environment variables:
#   INFURA_PROJECT_ID  — free account at https://app.infura.io
#   PRIVATE_KEY        — private key of the wallet holding the 203T tokens
#
# Run from terminal:
#   export INFURA_PROJECT_ID=your_key
#   export PRIVATE_KEY=your_private_key
# ---------------------------------------------------------------------------

INFURA_PROJECT_ID = os.environ.get("INFURA_PROJECT_ID")
PRIVATE_KEY       = os.environ.get("PRIVATE_KEY")

SENDER_ADDRESS    = "0x0390cF896B4a7D984017e6C9D3d17b5A6287a874"  # deployer
RECIPIENT_ADDRESS = "0x0390cF896B4a7D984017e6C9D3d17b5A6287a874"  # replace with recipient

if not INFURA_PROJECT_ID or not PRIVATE_KEY:
    print("[SKIP] INFURA_PROJECT_ID or PRIVATE_KEY not set.")
    print(f"       Would transfer {score:.2f} tokens (203T) to {RECIPIENT_ADDRESS}.")
else:
    tt = TokenTransfer(private_key=PRIVATE_KEY, sender_address=SENDER_ADDRESS)
    tx_hash = tt.transfer_tokens_from_rating(recipient=RECIPIENT_ADDRESS, rating=score)
    print(f"Transaction sent: {tx_hash}")
    print(f"https://sepolia.etherscan.io/tx/{tx_hash}")

---
## 4 · Explore the signal logic

Let's visualise how signals are generated for one ticker.

In [ ]:
from pybacktestchain.data_module import get_stocks_data

raw  = get_stocks_data(["AAPL"], datetime(2023, 1, 1), datetime(2023, 12, 31))
data = bt.compute_signals(raw)
aapl = data[data["ticker"] == "AAPL"].copy()

signal_counts = aapl["Position"].value_counts().rename({1: "Buy", -1: "Sell", 0: "Hold"})
print("AAPL signal distribution:")
print(signal_counts.to_string())
print(f"\nTotal trading days : {len(aapl)}")

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    fig.suptitle("AAPL — Volume signals (2023)", fontsize=14, fontweight="bold")

    # Price
    ax1.plot(aapl["Date"], aapl["Adj Close"], color="steelblue", linewidth=1.5)
    buys  = aapl[aapl["Position"] ==  1]
    sells = aapl[aapl["Position"] == -1]
    ax1.scatter(buys["Date"],  buys["Adj Close"],  marker="^", color="green",  s=60, zorder=5, label="Buy")
    ax1.scatter(sells["Date"], sells["Adj Close"], marker="v", color="red",    s=60, zorder=5, label="Sell")
    ax1.set_ylabel("Adj Close (USD)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Volume
    colors = aapl["Position"].map({1: "green", -1: "red", 0: "steelblue"})
    ax2.bar(aapl["Date"], aapl["Volume"] / 1e6, color=colors, alpha=0.7, width=1)
    ax2.plot(aapl["Date"], aapl["Avg Volume"] / 1e6, color="orange",
             linewidth=1.5, linestyle="--", label="15-day avg")
    ax2.plot(aapl["Date"], aapl["Avg Volume"] * 1.5 / 1e6, color="green",
             linewidth=1, linestyle=":", label="×1.5 threshold (buy)")
    ax2.plot(aapl["Date"], aapl["Avg Volume"] * 0.5 / 1e6, color="red",
             linewidth=1, linestyle=":", label="×0.5 threshold (sell)")
    ax2.set_ylabel("Volume (M shares)")
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib not installed — install it with: pip install matplotlib")

---
## 5 · Inspect the token on-chain

Read-only — no private key needed, only `INFURA_PROJECT_ID`.

In [ ]:
if not INFURA_PROJECT_ID:
    print("[SKIP] INFURA_PROJECT_ID not set.")
else:
    from web3 import Web3

    CONTRACT_ADDRESS = "0xA0f0a2D53b3476c50F2Cf24307F8a1Cd3c758254"
    DEPLOYER_ADDRESS = "0x0390cF896B4a7D984017e6C9D3d17b5A6287a874"

    ABI = [
        {"inputs": [],"name": "name","outputs": [{"type": "string"}],"stateMutability": "view","type": "function"},
        {"inputs": [],"name": "symbol","outputs": [{"type": "string"}],"stateMutability": "view","type": "function"},
        {"inputs": [],"name": "totalSupply","outputs": [{"type": "uint256"}],"stateMutability": "view","type": "function"},
        {"inputs": [{"name": "account","type": "address"}],"name": "balanceOf","outputs": [{"type": "uint256"}],"stateMutability": "view","type": "function"},
    ]

    w3       = Web3(Web3.HTTPProvider(f"https://sepolia.infura.io/v3/{INFURA_PROJECT_ID}"))
    contract = w3.eth.contract(address=CONTRACT_ADDRESS, abi=ABI)

    name         = contract.functions.name().call()
    symbol       = contract.functions.symbol().call()
    total_supply = float(w3.from_wei(contract.functions.totalSupply().call(), "ether"))
    balance      = float(w3.from_wei(contract.functions.balanceOf(DEPLOYER_ADDRESS).call(), "ether"))

    print(f"Token        : {name} ({symbol})")
    print(f"Total supply : {total_supply:.0f} {symbol}")
    print(f"Deployer bal : {balance:.4f} {symbol}")
    print(f"\nhttps://sepolia.etherscan.io/token/{CONTRACT_ADDRESS}")